In [ ]:
# Packages
library(ggplot2)
library(dplyr)
library(jsonlite)
library(data.table)
library(stringr)

# Charger les paramètres depuis un JSON Galaxy
json_input <- fromJSON("galaxy_inputs/galaxy_inputs.json")

# Extraire les paramètres
data_path         <- json_input$data$path
col_Y             <- json_input$col_Y             # Exemple: "Cover"
col_groupe_line   <- json_input$col_groupe_line   # Exemple: "site"
col_groupe_point  <- json_input$col_groupe_point  # Exemple: "sp richness"
plot_title        <- json_input$title
x_axis_label      <- json_input$x
y_axis_label      <- json_input$y
w                 <- json_input$width

output_path       <- "outputs/collection/plot.png"
# Lire les données
df <- fread(data_path)

# Nettoyage noms de colonnes
colnames(df) <- make.names(colnames(df))

# Renommer les colonnes selon les paramètres
year_col <- "Year"
y_col    <- make.names(col_Y)
group_l  <- make.names(col_groupe_line)
group_p  <- make.names(col_groupe_point)


# Grouper et faire les moyennes (si nécessaire)
agg_df <- df %>%
  group_by(.data[[year_col]], .data[[group_l]]) %>%
  summarise(
    Y = mean(.data[[y_col]], na.rm = TRUE),
    Grp_point = mean(.data[[group_p]], na.rm = TRUE),
    .groups = "drop"
  )

# Plot
png(output_path, width = w)

ggplot(agg_df, aes(x = .data[[year_col]], y = Y, color = .data[[group_l]], group = .data[[group_l]])) +
  geom_line(size = 1) +
  geom_point(aes(size = Grp_point), alpha = 0.7) +
  labs(
    title = plot_title,
    x = x_axis_label,
    y = y_axis_label,
    color = group_l,
    size = group_p
  ) +
  theme_minimal()

dev.off()

